# Building Indoor Climate Simulation

This notebook simulates indoor temperature and humidity for an office building based on outdoor meteorological data and HVAC system behavior.

## Project Overview
- **Input**: Preprocessed outdoor weather data (2020-2025) with HVAC setpoints
- **Output**: Simulated indoor temperature and humidity
- **Goal**: Train a model to predict future indoor conditions based on weather forecasts

## Building Thermal Model
We'll use a simplified thermal model that considers:
- Heat transfer through building envelope
- HVAC system operation
- Internal heat gains (occupancy, equipment, lighting)
- Thermal mass effects


In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.integrate import solve_ivp
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


In [6]:
# Load the preprocessed data
df = pd.read_csv(r'C:\Users\SunnyVanderwall\OneDrive - Medieinstitutet i Sverige\Documents\time_series_summer_ml\final_df_3.csv', index_col=0)
print(f"Data shape: {df.shape}")
print(f"Date range: {df['Datetime'].min()} to {df['Datetime'].max()}")
df.head()


Data shape: (48192, 24)
Date range: 2020-01-01 00:00:00 to 2025-06-30 23:00:00


,Time (UTC),Datetime,Air Temperature,Air Temperature Quality,Precipitation Amount,Precipitation Amount Quality,Wind Direction,Wind Direction Quality,Wind Speed,Wind Speed Quality,...,Sunshine Duration (s),Sunshine Duration Quality,Hour sin,Hour cos,Day of Year sin,Day of Year cos,Month sin,Month cos,HVAC Mode,Temp Set Point
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-01,00:00:00,2020-01-01 00:00:00,2.0,0.0,6.938894e-18,1.355253e-20,250.0,0.0,3.0,-1.355253e-20,...,0.0,0.0,0.000000,1.000000,0.017213,0.999852,0.5,0.866025,Heating,17
2020-01-01,01:00:00,2020-01-01 01:00:00,2.2,0.0,6.938894e-18,1.355253e-20,250.0,0.0,3.0,-1.355253e-20,...,0.0,0.0,0.258819,0.965926,0.017213,0.999852,0.5,0.866025,Heating,17
2020-01-01,02:00:00,2020-01-01 02:00:00,2.2,0.0,6.938894e-18,1.355253e-20,240.0,0.0,3.0,-1.355253e-20,...,0.0,0.0,0.500000,0.866025,0.017213,0.999852,0.5,0.866025,Heating,17
2020-01-01,03:00:00,2020-01-01 03:00:00,1.6,0.0,6.938894e-18,1.355253e-20,240.0,0.0,3.0,-1.355253e-20,...,0.0,0.0,0.707107,0.707107,0.017213,0.999852,0.5,0.866025,Heating,17
2020-01-01,04:00:00,2020-01-01 04:00:00,1.5,0.0,6.938894e-18,1.355253e-20,230.0,0.0,3.0,-1.355253e-20,...,0.0,0.0,0.866025,0.500000,0.017213,0.999852,0.5,0.866025,Heating,17


## Building Thermal Model Parameters

We'll define a simplified office building with typical thermal characteristics:


In [4]:
# Building thermal parameters (typical office building)
class BuildingParameters:
    def __init__(self):
        # Building geometry
        self.floor_area = 1000  # m²
        self.ceiling_height = 3.0  # m
        self.volume = self.floor_area * self.ceiling_height  # m³
        
        # Thermal properties
        self.U_wall = 0.3  # W/m²K (U-value for walls)
        self.U_roof = 0.25  # W/m²K (U-value for roof)
        self.U_window = 1.2  # W/m²K (U-value for windows)
        
        # Surface areas (assuming square building)
        self.wall_area = 4 * np.sqrt(self.floor_area) * self.ceiling_height  # m²
        self.roof_area = self.floor_area  # m²
        self.window_area = 0.2 * self.wall_area  # 20% of wall area
        
        # Thermal mass
        self.thermal_mass = 200000  # J/K (building thermal mass)
        
        # HVAC system
        self.hvac_capacity = 50000  # W (heating/cooling capacity)
        self.hvac_efficiency = 0.8  # efficiency factor
        
        # Internal heat gains
        self.occupancy_heat = 100  # W/person
        self.equipment_heat = 15  # W/m²
        self.lighting_heat = 10  # W/m²
        self.occupancy_density = 0.1  # persons/m²
        
        # Air properties
        self.air_density = 1.225  # kg/m³
        self.air_heat_capacity = 1006  # J/kgK
        self.ventilation_rate = 0.5  # air changes per hour

# Initialize building parameters
building = BuildingParameters()
print("Building Parameters:")
print(f"Floor area: {building.floor_area} m²")
print(f"Volume: {building.volume} m³")
print(f"Thermal mass: {building.thermal_mass} J/K")
print(f"HVAC capacity: {building.hvac_capacity} W")


Building Parameters:
Floor area: 1000 m²
Volume: 3000.0 m³
Thermal mass: 200000 J/K
HVAC capacity: 50000 W


## Thermal Model Implementation

The building thermal model uses a simplified heat balance equation:

**dT/dt = (Q_solar + Q_internal + Q_hvac - Q_transmission - Q_ventilation) / C_thermal**

Where:
- Q_solar: Solar heat gains through windows
- Q_internal: Internal heat gains (occupancy, equipment, lighting)
- Q_hvac: HVAC heating/cooling
- Q_transmission: Heat loss through building envelope
- Q_ventilation: Heat loss through ventilation
- C_thermal: Building thermal mass


In [5]:
class BuildingThermalModel:
    def __init__(self, building_params):
        self.building = building_params
        
    def calculate_solar_gains(self, solar_irradiance, hour):
        """Calculate solar heat gains through windows"""
        # Solar heat gain coefficient for windows
        SHGC = 0.6
        
        # Solar angle factor (simplified - varies with time of day)
        # Peak at noon (hour 12), minimum at night
        solar_angle_factor = max(0, np.sin(np.pi * (hour - 6) / 12)) if 6 <= hour <= 18 else 0
        
        solar_gains = solar_irradiance * self.building.window_area * SHGC * solar_angle_factor
        return solar_gains
    
    def calculate_internal_gains(self, hour, is_weekday=True):
        """Calculate internal heat gains from occupancy, equipment, and lighting"""
        # Occupancy pattern (higher during office hours)
        if is_weekday and 8 <= hour <= 18:
            occupancy_factor = 1.0
        elif is_weekday and (7 <= hour < 8 or 18 < hour <= 19):
            occupancy_factor = 0.5
        else:
            occupancy_factor = 0.1
            
        # Equipment and lighting (on during occupied hours)
        equipment_factor = 1.0 if occupancy_factor > 0.5 else 0.3
        lighting_factor = 1.0 if occupancy_factor > 0.5 else 0.1
        
        occupancy_gains = (self.building.occupancy_heat * 
                          self.building.occupancy_density * 
                          self.building.floor_area * occupancy_factor)
        
        equipment_gains = (self.building.equipment_heat * 
                          self.building.floor_area * equipment_factor)
        
        lighting_gains = (self.building.lighting_heat * 
                         self.building.floor_area * lighting_factor)
        
        return occupancy_gains + equipment_gains + lighting_gains
    
    def calculate_hvac_load(self, indoor_temp, setpoint_temp, outdoor_temp, hvac_mode):
        """Calculate HVAC heating/cooling load"""
        temp_diff = setpoint_temp - indoor_temp
        
        if hvac_mode == 'Heating' and temp_diff > 0.5:
            # Heating needed
            hvac_load = min(self.building.hvac_capacity, 
                           temp_diff * self.building.thermal_mass * 0.1)
        elif hvac_mode == 'Cooling' and temp_diff < -0.5:
            # Cooling needed
            hvac_load = -min(self.building.hvac_capacity, 
                            abs(temp_diff) * self.building.thermal_mass * 0.1)
        else:
            hvac_load = 0
            
        return hvac_load * self.building.hvac_efficiency
    
    def calculate_transmission_loss(self, indoor_temp, outdoor_temp):
        """Calculate heat loss through building envelope"""
        temp_diff = indoor_temp - outdoor_temp
        
        # Total U-value weighted by area
        total_UA = (self.building.U_wall * (self.building.wall_area - self.building.window_area) +
                   self.building.U_roof * self.building.roof_area +
                   self.building.U_window * self.building.window_area)
        
        transmission_loss = total_UA * temp_diff
        return transmission_loss
    
    def calculate_ventilation_loss(self, indoor_temp, outdoor_temp, outdoor_humidity):
        """Calculate heat loss through ventilation"""
        temp_diff = indoor_temp - outdoor_temp
        
        # Ventilation heat loss
        ventilation_flow = (self.building.ventilation_rate * 
                           self.building.volume / 3600)  # m³/s
        
        ventilation_loss = (ventilation_flow * 
                           self.building.air_density * 
                           self.building.air_heat_capacity * 
                           temp_diff)
        
        return ventilation_loss
    
    def thermal_balance(self, t, T_indoor, outdoor_temp, solar_irradiance, 
                       setpoint_temp, hvac_mode, hour, outdoor_humidity):
        """Thermal balance equation for indoor temperature"""
        
        # Calculate all heat flows
        solar_gains = self.calculate_solar_gains(solar_irradiance, hour)
        internal_gains = self.calculate_internal_gains(hour)
        hvac_load = self.calculate_hvac_load(T_indoor, setpoint_temp, outdoor_temp, hvac_mode)
        transmission_loss = self.calculate_transmission_loss(T_indoor, outdoor_temp)
        ventilation_loss = self.calculate_ventilation_loss(T_indoor, outdoor_temp, outdoor_humidity)
        
        # Thermal balance equation
        dT_dt = (solar_gains + internal_gains + hvac_load - 
                transmission_loss - ventilation_loss) / self.building.thermal_mass
        
        return dT_dt

# Initialize thermal model
thermal_model = BuildingThermalModel(building)
